# 🔗 Bölüm 3: Koşullu Olasılık ve Bağımsızlık
## Mühendisler İçin Olasılık

**Konular:**
1. Koşullu Olasılık — Tanım ve Örnekler
2. Çarpma Kuralı — Olasılıkların Zincirlenmesi
3. Toplam Olasılık Yasası
4. Bayes Teoremi
5. Bağımsız Olaylar
6. Karşılıklı Bağımsızlık
7. Olasılık Algıları ve Önyargılar (Linda Problemi)

---

In [ ]:
import math
import itertools
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle, FancyArrow
from fractions import Fraction

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

print("Kütüphaneler yüklendi ✓")

---
## 1. Koşullu Olasılık

**Tanım:** $P(B) > 0$ olan iki olay $A$ ve $B$ için, $B$ gerçekleşmiş olduğunda $A$'nın koşullu olasılığı:

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}$$

**Sezgi (eşit olasılıklı uzay):** $P(A \mid B) = \dfrac{\#(A \cap B)}{\#(B)}$

Yani B'yi yeni örneklem uzayı kabul ediyoruz ve A'nın bu kümedeki payını hesaplıyoruz.

In [ ]:
def kosullu_olasilik(p_A_ve_B, p_B):
    """P(A|B) = P(A∩B) / P(B)"""
    assert p_B > 0, "P(B) > 0 olmalı!"
    return p_A_ve_B / p_B

def kosullu_esit(S, A, B):
    """Eşit olasılıklı uzayda P(A|B) = |A∩B| / |B|"""
    kesisim = A & B
    return len(kesisim) / len(B) if len(B) > 0 else 0

# ──────────────────────────────────────────
# Örnek 1: Tek sayı koşulunda 1 gelme
S_zar = set(range(1, 7))
A = {1}              # A = {1 gelme}
B = {1, 3, 5}        # B = {tek sayı}

p_A   = len(A) / len(S_zar)
p_B   = len(B) / len(S_zar)
p_AnB = len(A & B) / len(S_zar)
p_AgB = kosullu_esit(S_zar, A, B)

print("Örnek 1: Zar — Tek sayı koşulunda 1 gelme")
print(f"  S = {sorted(S_zar)}")
print(f"  A = {sorted(A)}  (1 gelme),  P(A) = {p_A:.4f} = {Fraction(len(A),len(S_zar))}")
print(f"  B = {sorted(B)}  (tek sayı), P(B) = {p_B:.4f} = {Fraction(len(B),len(S_zar))}")
print(f"  A∩B = {sorted(A&B)},  P(A∩B) = {p_AnB:.4f} = {Fraction(len(A&B),len(S_zar))}")
print(f"  P(A|B) = P(A∩B)/P(B) = {p_AnB:.4f}/{p_B:.4f} = {p_AgB:.4f} = {Fraction(len(A&B),len(B))}")
print(f"  Yorum: Tek sayı geldiğini BİLİYORUZ → sadece {{1,3,5}} içinden seçim")

print()

# Örnek 2: İki zar — toplam 7, ilk atış 4 koşulunda
S_iki = [(i,j) for i in range(1,7) for j in range(1,7)]
A2 = {(i,j) for i,j in S_iki if i+j == 7}   # Toplam 7
B2 = {(i,j) for i,j in S_iki if i == 4}      # İlk atış 4

p_A2gB2 = kosullu_esit(set(S_iki), A2, B2)

print("Örnek 2: İki Zar — Toplam 7, ilk atış 4 koşulunda")
print(f"  A = {{Toplam 7}} = {sorted(A2)}")
print(f"  B = {{İlk atış 4}} = {sorted(B2)}")
print(f"  A∩B = {sorted(A2&B2)}  ← sadece (4,3)")
print(f"  P(A|B) = {len(A2&B2)}/{len(B2)} = {p_A2gB2:.4f} = {Fraction(len(A2&B2),len(B2))}")

In [ ]:
# Koşullu olasılık görselleştirmesi — örneklem uzayının daralması
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sol: tüm zar uzayı
ax = axes[0]
ax.set_aspect('equal'); ax.set_title('Koşulsuz: P(A=1) = 1/6', fontsize=12, fontweight='bold')
ax.set_xlim(0, 7); ax.set_ylim(0, 2)
ax.axis('off')
renkler = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f0e','#a65628']
for i, n in enumerate(range(1, 7)):
    renk = '#e41a1c' if n == 1 else '#cccccc'
    c = Circle((i+0.8, 1), 0.38, color=renk, alpha=0.8, zorder=2)
    ax.add_patch(c)
    ax.text(i+0.8, 1, str(n), ha='center', va='center', fontsize=14,
            fontweight='bold', color='white' if n==1 else 'black')
ax.text(3.5, 0.2, 'S = {1,2,3,4,5,6}  →  P(1) = 1/6', ha='center', fontsize=10)

# Sağ: B={tek} koşulunda daralma
ax2 = axes[1]
ax2.set_aspect('equal'); ax2.set_title('Koşullu: P(A=1 | B=tek) = 1/3', fontsize=12, fontweight='bold')
ax2.set_xlim(0, 7); ax2.set_ylim(0, 2)
ax2.axis('off')
for i, n in enumerate([1, 3, 5]):
    renk = '#e41a1c' if n == 1 else '#4daf4a'
    c = Circle((i*2+1, 1), 0.38, color=renk, alpha=0.85, zorder=2)
    ax2.add_patch(c)
    ax2.text(i*2+1, 1, str(n), ha='center', va='center', fontsize=14,
             fontweight='bold', color='white')
# Silik (dışlanan)
for i, n in enumerate([2, 4, 6]):
    c = Circle((i*2+0.3, 0.3), 0.22, color='#dddddd', alpha=0.5, zorder=1)
    ax2.add_patch(c)
    ax2.text(i*2+0.3, 0.3, str(n), ha='center', va='center', fontsize=9, color='#aaaaaa')
ax2.text(3.5, 1.7, 'B = {tek} yeni örneklem uzayı', ha='center', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='#ffffcc'))
ax2.text(3.5, 0.1, 'P(1|tek) = 1/3', ha='center', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='#ffcccc'))

plt.suptitle('Koşullu Olasılık — Örneklem Uzayının Daralması',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Koşullu olasılığın özellikleri — sayısal doğrulama
print("Koşullu Olasılığın Önermeleri")
print("=" * 45)

S = set(range(1, 7))
A = {2, 4, 6};  B = {1, 2, 3, 4}

def p(E, uzay=S):  return len(E & uzay) / len(uzay)

# Önerme 1: P(A|A) = 1
p_AgA = kosullu_esit(S, A, A)
print(f"1. P(A|A) = {p_AgA:.4f}  {'✓' if abs(p_AgA-1)<1e-9 else '✗'}")

# Önerme 2: P(Aᶜ|A) = 0
Ac = S - A
p_AcgA = kosullu_esit(S, Ac, A)
print(f"2. P(Aᶜ|A) = {p_AcgA:.4f}  {'✓' if abs(p_AcgA)<1e-9 else '✗'}")

# Önerme 3: P(Aᶜ|B) = 1 - P(A|B)
p_AgB  = kosullu_esit(S, A, B)
p_AcgB = kosullu_esit(S, Ac, B)
print(f"3. P(Aᶜ|B) = {p_AcgB:.4f},  1-P(A|B) = {1-p_AgB:.4f}  {'✓' if abs(p_AcgB-(1-p_AgB))<1e-9 else '✗'}")
print()

# Koşullu olasılık tablosu — tüm zar çiftleri
print("P(Toplam=k | İlk atış = i) matrisi:")
print(f"{'':>4}", end="")
for k in range(2, 13):
    print(f"{k:>6}", end="")
print()
print("-" * 72)

for i in range(1, 7):
    B_ilk = {(a, b) for a, b in S_iki if a == i}  # ilk atış i olan tüm çiftler
    print(f"i={i} |", end="")
    for k in range(2, 13):
        A_top = {(a,b) for a,b in S_iki if a+b==k}
        pval = len(A_top & B_ilk) / len(B_ilk)
        print(f"  {pval:.2f}", end="")
    print()

---
## 2. Çarpma Kuralı — Olasılıkların Zincirlenmesi

$$P(A \cap B) = P(A \mid B) \cdot P(B)$$

**Genelleştirme (Zincir Kuralı):**
$$P\!\left(\bigcap_{i=1}^n A_i\right) = P(A_1)\cdot P(A_2\mid A_1)\cdot P(A_3\mid A_1,A_2)\cdots P(A_n\mid A_1,\ldots,A_{n-1})$$

In [ ]:
# Kutudaki top örneği — yerine koymadan çekme
# 8 beyaz (W) + 4 siyah (B) top, 3 çekiliş

def carpma_kurali_top(n_beyaz, n_siyah, sira):
    """
    sira: ör. ['W','B','B'] → W=Beyaz(White), B=Siyah(Black)
    Yerine koymadan çekme — çarpma kuralı
    """
    n_toplam = n_beyaz + n_siyah
    n_b, n_s = n_beyaz, n_siyah
    olasilik = 1.0
    adimlar  = []
    for renk in sira:
        if renk == 'W':
            p = n_b / n_toplam
            adimlar.append((f'P(W|mevcut:{n_b}B,{n_s}S)', p, Fraction(n_b, n_toplam)))
            n_b     -= 1
        else:   # 'B' (Siyah/Black)
            p = n_s / n_toplam
            adimlar.append((f'P(S|mevcut:{n_b}B,{n_s}S)', p, Fraction(n_s, n_toplam)))
            n_s     -= 1
        n_toplam -= 1
        olasilik *= p
    return olasilik, adimlar

print("Kutu: 8 Beyaz + 4 Siyah top — Yerine koymadan 3 çekiliş")
print("=" * 58)

# Soru 1: W1 B2 B3
p1, adimlar1 = carpma_kurali_top(8, 4, ['W', 'B', 'B'])
print("\nSoru 1: P(W₁ B₂ B₃) = ?  (Çarpma Kuralı)")
for i, (aciklama, p, frac) in enumerate(adimlar1, 1):
    print(f"  Adım {i}: {aciklama} = {frac} = {p:.5f}")
frac_sonuc = Fraction(8,12) * Fraction(4,11) * Fraction(3,10)
print(f"  P(W₁B₂B₃) = 8/12 × 4/11 × 3/10 = {frac_sonuc} = {p1:.6f}")

print()

# Soru 2: P(B₂) — 2. çekilişte siyah
p_W1B2, _ = carpma_kurali_top(8, 4, ['W', 'B'])
p_B1B2, _ = carpma_kurali_top(8, 4, ['B', 'B'])
p_B2 = p_W1B2 + p_B1B2

f_W1B2 = Fraction(8,12) * Fraction(4,11)
f_B1B2 = Fraction(4,12) * Fraction(3,11)
f_B2   = f_W1B2 + f_B1B2

print("Soru 2: P(B₂) = ?  (1. çekiliş bilinmiyor)")
print(f"  P(W₁B₂) = 8/12 × 4/11 = {f_W1B2} = {p_W1B2:.6f}")
print(f"  P(B₁B₂) = 4/12 × 3/11 = {f_B1B2} = {p_B1B2:.6f}")
print(f"  P(B₂) = P(W₁B₂) + P(B₁B₂) = {f_B2} = {p_B2:.6f}")
print(f"  Not: P(B₂) = P(B₁) = 4/12 = {4/12:.6f}  (simetri!)")

In [ ]:
# Zincir kuralı — ağaç diyagramı görselleştirmesi
fig, ax = plt.subplots(figsize=(13, 7))
ax.set_xlim(-0.5, 5.5)
ax.set_ylim(-0.5, 7)
ax.axis('off')
ax.set_title('Çarpma Kuralı — Ağaç Diyagramı (8W + 4S, 3 Çekiliş)',
             fontsize=13, fontweight='bold')

# Kök
kok_x, kok_y = 0.3, 3.5
ax.text(kok_x, kok_y, 'Başlangıç\n(8W+4S)', ha='center', va='center', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#f0f0f0'), zorder=3)

# 1. Seviye
dal1 = [('W', 1.5, 5.5, '8/12', '#92c5de'), ('S', 1.5, 1.5, '4/12', '#f4a582')]
for renk, x, y, frac, col in dal1:
    ax.annotate('', xy=(x-0.2, y), xytext=(kok_x+0.2, kok_y),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.3))
    ax.text(x, y, f'{renk}₁', ha='center', va='center', fontsize=10,
            fontweight='bold',
            bbox=dict(boxstyle='round', facecolor=col, alpha=0.8), zorder=3)
    orta_x = (kok_x + x) / 2 + 0.1
    orta_y = (kok_y + y) / 2
    ax.text(orta_x, orta_y, frac, fontsize=8, color='navy', ha='center')

# 2. Seviye
dal2 = [
    ('W', 1.5, 5.5, 'W', 2.8, 6.3, '7/11', '#92c5de'),
    ('W', 1.5, 5.5, 'S', 2.8, 4.7, '4/11', '#f4a582'),
    ('S', 1.5, 1.5, 'W', 2.8, 2.3, '8/11', '#92c5de'),
    ('S', 1.5, 1.5, 'S', 2.8, 0.7, '3/11', '#f4a582'),
]
for _, x1, y1, renk2, x2, y2, frac2, col in dal2:
    ax.annotate('', xy=(x2-0.2, y2), xytext=(x1+0.2, y1),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.1))
    ax.text(x2, y2, f'{renk2}₂', ha='center', va='center', fontsize=9,
            bbox=dict(boxstyle='round', facecolor=col, alpha=0.7), zorder=3)
    ax.text((x1+x2)/2+0.1, (y1+y2)/2, frac2, fontsize=7, color='navy', ha='center')

# 3. Seviye — sadece W1B2B3 yolunu vurgula
vurgulu = [  # (baba_x, baba_y, renk3, x3, y3, frac3)
    (2.8, 4.7, 'S', 4.2, 4.2, '3/10', True),
    (2.8, 0.7, 'S', 4.2, 0.3, '2/10', False),
    (2.8, 6.3, 'S', 4.2, 6.0, '4/10', False),
]
for x1, y1, renk3, x2, y2, frac3, vurgula in vurgulu:
    stil  = '#ff7f00' if vurgula else '#f4a582'
    lw    = 2.5       if vurgula else 1.0
    ax.annotate('', xy=(x2-0.2, y2), xytext=(x1+0.2, y1),
                arrowprops=dict(arrowstyle='->', color=stil, lw=lw))
    ax.text(x2, y2, f'{renk3}₃', ha='center', va='center', fontsize=9,
            bbox=dict(boxstyle='round', facecolor=stil, alpha=0.8), zorder=3)
    ax.text((x1+x2)/2+0.1, (y1+y2)/2, frac3, fontsize=7, color='navy', ha='center')

# Vurgulanan yol etiketi
ax.text(4.8, 4.2, f'W₁B₂B₃\n= 8/12×4/11×3/10\n= {Fraction(8,12)*Fraction(4,11)*Fraction(3,10)}',
        ha='left', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#ffffcc', edgecolor='orange', lw=1.5))

plt.tight_layout()
plt.show()

---
## 3. Toplam Olasılık Yasası

$A_1, \ldots, A_n$ ayrık ve $\bigcup_{i=1}^n A_i = S$ (**bölümleme**) ise:

$$P(B) = \sum_{j=1}^n P(B \mid A_j)\, P(A_j)$$

In [ ]:
def toplam_olasilik(p_Ai_listesi, p_BgAi_listesi):
    """P(B) = Σ P(B|Ai) * P(Ai)"""
    assert abs(sum(p_Ai_listesi) - 1.0) < 1e-9, "P(Ai) toplamı 1 olmalı!"
    return sum(prior * lik for prior, lik in zip(p_Ai_listesi, p_BgAi_listesi))

# ──────────────────────────────────────────
# Randomize Yanıt Anketi (uyuşturucu örneği)
print("=" * 55)
print("Randomize Yanıt Anketi — Uyuşturucu Sorusu")
print("=" * 55)

# A1 = {tura}, A2 = {yazı}
p_A1 = 0.5  # P(tura)
p_A2 = 0.5  # P(yazı)

# P(EVET|tura) = P(araba paylaşımı) = 0.35
# P(EVET|yazı) = P(uyuşturucu) = ?
p_evet_gA1  = 0.35    # araba paylaşımı oranı (biliniyor)
p_evet_gozl = 1420 / 8000  # gözlemlenen EVET oranı

# Toplam olasılık: P(EVET) = P(EVET|A1)*P(A1) + P(EVET|A2)*P(A2)
# p_evet_gozl = p_evet_gA1 * p_A1 + p_uyu * p_A2
p_uyusturucu = (p_evet_gozl - p_evet_gA1 * p_A1) / p_A2

print(f"\nGözlemlenen: {1420}/{8000} = {p_evet_gozl:.5f} EVET yanıtı")
print(f"P(EVET|tura) = P(araba paylaşımı) = {p_evet_gA1}")
print(f"P(EVET) = P(EVET|tura)×P(tura) + P(uyuşturucu)×P(yazı)")
print(f"{p_evet_gozl:.5f} = {p_evet_gA1} × {p_A1} + P(uyuşturucu) × {p_A2}")
print(f"P(uyuşturucu) = ({p_evet_gozl:.5f} - {p_evet_gA1*p_A1}) / {p_A2} = {p_uyusturucu:.4f}")
print(f"\nSonuç: Çalışanların yaklaşık %{p_uyusturucu*100:.1f}'i uyuşturucu kullanıyor")

# Doğrulama
p_kontrol = toplam_olasilik([p_A1, p_A2], [p_evet_gA1, p_uyusturucu])
print(f"\nDoğrulama: P(EVET) = {p_kontrol:.5f} ≈ {p_evet_gozl:.5f}  {'✓' if abs(p_kontrol-p_evet_gozl)<1e-6 else '✗'}")

In [ ]:
# Toplam olasılık görselleştirmesi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: Randomize anket — ağaç
ax = axes[0]
ax.set_xlim(-0.3, 3); ax.set_ylim(0, 4); ax.axis('off')
ax.set_title('Randomize Yanıt — Toplam Olasılık', fontsize=11, fontweight='bold')

# Kök
ax.text(0.1, 2, 'Çalışan\nYazı-Tura Atar', ha='center', va='center', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#f0f0f0'))

dallar = [
    ('Tura (0.5)', 1.2, 3.2, 'Araba\nPaylaşımı?', '#92c5de', 0.35, '#4daf4a', '#e41a1c'),
    ('Yazı (0.5)', 1.2, 0.8, 'Uyuşturucu\nKullandın mı?', '#f4a582', p_uyusturucu, '#4daf4a', '#e41a1c'),
]
for etiket, x1, y1, soru, col, p_evet, col_e, col_h in dallar:
    ax.annotate('', xy=(x1-0.15, y1), xytext=(0.3, 2),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.3))
    ax.text(x1, y1+0.3, etiket, ha='center', fontsize=8, color='navy')
    ax.text(x1, y1, soru, ha='center', va='center', fontsize=8,
            bbox=dict(boxstyle='round', facecolor=col, alpha=0.7))
    ax.text(2.4, y1+0.3, f'EVET ({p_evet:.3f})', ha='center', fontsize=8,
            bbox=dict(boxstyle='round', facecolor=col_e, alpha=0.6))
    ax.text(2.4, y1-0.3, f'HAYIR ({1-p_evet:.3f})', ha='center', fontsize=8,
            bbox=dict(boxstyle='round', facecolor=col_h, alpha=0.4))
    ax.annotate('', xy=(2.1, y1+0.3), xytext=(x1+0.3, y1),
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.9))
    ax.annotate('', xy=(2.1, y1-0.3), xytext=(x1+0.3, y1),
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.9))

ax.text(1.5, 0.1, f'P(EVET) = {p_kontrol:.4f}', ha='center', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='lightyellow'))

# Sağ: Bölümleme konsepti
ax2 = axes[1]
ax2.set_xlim(0, 10); ax2.set_ylim(0, 6); ax2.set_aspect('equal'); ax2.axis('off')
ax2.set_title('Toplam Olasılık: S Bölümlenmesi', fontsize=11, fontweight='bold')

rect = plt.Rectangle((0.3, 0.3), 9.4, 5.2, fill=False, edgecolor='black', linewidth=2)
ax2.add_patch(rect)
bolumler = [(0.3, 0.3, 2.5, 5.2, '#92c5de', 'A₁'), (2.8, 0.3, 3.5, 5.2, '#f4a582', 'A₂'),
            (6.3, 0.3, 2.0, 5.2, '#b2df8a', 'A₃'), (8.3, 0.3, 1.4, 5.2, '#984ea3', 'A₄')]
for x, y, w, h, col, lbl in bolumler:
    r = plt.Rectangle((x, y), w, h, color=col, alpha=0.4)
    ax2.add_patch(r)
    ax2.text(x + w/2, y + h/2 + 1.0, lbl, ha='center', va='center',
             fontsize=13, fontweight='bold', color=col)

# B olayı (yatay elips)
from matplotlib.patches import Ellipse
ell = Ellipse((5, 2.9), 8, 1.8, color='gold', alpha=0.55, zorder=2)
ax2.add_patch(ell)
ax2.text(5, 2.9, 'B', ha='center', va='center', fontsize=14,
         fontweight='bold', color='darkgoldenrod', zorder=3)

ax2.text(5, 0.0, 'P(B) = Σ P(B|Aᵢ)·P(Aᵢ)',
         ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='lightyellow'))
ax2.text(9.5, 5.3, 'S', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 4. Bayes Teoremi

$A_1,\ldots,A_n$ bölümleme, $P(A_i)>0$, $P(B)>0$ ise:

$$P(A_i \mid B) = \frac{P(B \mid A_i)\,P(A_i)}{\displaystyle\sum_{j=1}^n P(B \mid A_j)\,P(A_j)}$$

- $P(A_i)$: **önsel** (prior) olasılık
- $P(A_i \mid B)$: **sonsal** (posterior) olasılık  
- Amaç: koşullandırmayı **ters çevirmek** — $P(B|A)$ biliniyorsa $P(A|B)$ hesapla

In [ ]:
def bayes(p_prior, p_likelihood):
    """
    p_prior     : [P(A1), P(A2), ..., P(An)]  — önsel olasılıklar
    p_likelihood: [P(B|A1), ..., P(B|An)]     — olabilirlikler
    Döndürür   : [P(A1|B), ..., P(An|B)]      — sonsal olasılıklar
    """
    p_B = sum(l * p for l, p in zip(p_likelihood, p_prior))
    posterior = [(l * p) / p_B for l, p in zip(p_likelihood, p_prior)]
    return posterior, p_B

# ──────────────────────────────────────────
# Vassar Örneği
print("Vassar Öğrencileri — Bayes Teoremi")
print("=" * 50)

# Önsel olasılıklar — bölüm dağılımı
prior = [0.25, 0.55, 0.20]   # [matematik, müzik, ekonomi]
bolum = ['Matematik', 'Müzik', 'Ekonomi']

# P(randevu | bölüm)
p_randevu = [0.90, 0.50, 0.10]

posterior, p_B = bayes(prior, p_randevu)

print(f"{'Bölüm':<12} {'P(Aᵢ)':>8} {'P(B|Aᵢ)':>10} {'P(B|Aᵢ)×P(Aᵢ)':>16} {'P(Aᵢ|B)':>10}")
print("-" * 60)
for b, pri, lik, post in zip(bolum, prior, p_randevu, posterior):
    ortak = pri * lik
    print(f"{b:<12} {pri:>8.2f} {lik:>10.2f} {ortak:>16.4f} {post:>10.4f}")
print("-" * 60)
print(f"{'P(B) = ':>40} {p_B:>10.4f}")
print()
print(f"Randevu alan biri matematik öğrencisi olma olasılığı:")
print(f"P(Matematik|Randevu) = {posterior[0]:.4f} ≈ {posterior[0]*100:.1f}%")
print(f"\nÖnsel P(Matematik) = {prior[0]:.2f} → Sonsal P(Matematik|Randevu) = {posterior[0]:.4f}")
print(f"Randevu haberi matematiği daha olası kıldı! (0.25 → {posterior[0]:.2f})")

In [ ]:
# Tıbbi test örneği — Bayes'in pratik önemi
print("Tıbbi Test Örneği — Bayes Teoremi")
print("=" * 50)
print("Hastalık prevalansı: %1  |  Test hassasiyeti: %99  |  Test özgüllüğü: %95")
print()

p_hasta  = 0.01    # P(Hasta)
p_saglikli = 0.99  # P(Sağlıklı)
p_poz_ghasta    = 0.99  # P(Test+ | Hasta)
p_poz_gsaglikli = 0.05  # P(Test+ | Sağlıklı) = 1 - özgüllük

prior_t = [p_hasta, p_saglikli]
lik_t   = [p_poz_ghasta, p_poz_gsaglikli]
post_t, p_test_poz = bayes(prior_t, lik_t)

print(f"P(Hasta | Test+)    = {post_t[0]:.4f} = {post_t[0]*100:.1f}%")
print(f"P(Sağlıklı | Test+) = {post_t[1]:.4f} = {post_t[1]*100:.1f}%")
print(f"\nSürpriz: Test+ çıkmasına rağmen hasta olma olasılığı yalnızca %{post_t[0]*100:.1f}!")
print(f"Neden? Prevalans düşük ({p_hasta*100}%) → yanlış pozitifler gerçek pozitifleri geçiyor")

# Prevalans farklı değerler için P(Hasta|Test+)
print()
print(f"{'Prevalans':>12} | {'P(Hasta|Test+)':>16}")
print("-" * 32)
for prev in [0.001, 0.01, 0.05, 0.10, 0.20, 0.50]:
    p_sag = 1 - prev
    posts, _ = bayes([prev, p_sag], [p_poz_ghasta, p_poz_gsaglikli])
    print(f"{prev*100:>10.1f}%  | {posts[0]*100:>14.2f}%")

In [ ]:
# Bayes güncellemesi görselleştirmesi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: Vassar — önsel vs sonsal
ax = axes[0]
x  = np.arange(len(bolum))
w  = 0.35
b1 = ax.bar(x - w/2, prior,     w, label='Önsel P(Aᵢ)',     color='#4C72B0', alpha=0.8, edgecolor='black', lw=0.5)
b2 = ax.bar(x + w/2, posterior, w, label='Sonsal P(Aᵢ|Randevu)', color='#DD8452', alpha=0.8, edgecolor='black', lw=0.5)
for bar, v in zip(b1, prior):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.2f}', ha='center', va='bottom', fontsize=9)
for bar, v in zip(b2, posterior):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.2f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(bolum, fontsize=11)
ax.set_ylabel('Olasılık', fontsize=11)
ax.set_title('Vassar: Önsel → Sonsal (Bayes Güncellemesi)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y'); ax.set_ylim(0, 0.75)

# Sağ: Prevalans vs P(Hasta|Test+)
ax2 = axes[1]
prevs = np.linspace(0.001, 0.5, 200)
post_hasta = []
for prev in prevs:
    p_sag = 1 - prev
    p_B_test = p_poz_ghasta * prev + p_poz_gsaglikli * p_sag
    post_hasta.append(p_poz_ghasta * prev / p_B_test)

ax2.plot(prevs * 100, [p*100 for p in post_hasta], color='#d62728', linewidth=2.5)
ax2.axhline(50, color='gray', linestyle='--', alpha=0.6, label='%50 eşiği')
ax2.scatter([1], [post_t[0]*100], color='red', s=120, zorder=5)
ax2.annotate(f'Prevalans=%1\nP(Hasta|+)={post_t[0]*100:.1f}%',
             xy=(1, post_t[0]*100), xytext=(8, 20),
             arrowprops=dict(arrowstyle='->', color='black'),
             fontsize=9, bbox=dict(boxstyle='round', facecolor='lightyellow'))
ax2.set_xlabel('Hastalık Prevalansı (%)', fontsize=11)
ax2.set_ylabel('P(Hasta | Test+) (%)', fontsize=11)
ax2.set_title('Bayes: Prevalans P(Hasta|Test+)\'i Nasıl Etkiler?', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Bağımsız Olaylar

**Tanım:** $A$ ve $B$ bağımsızdır, eğer:
$$P(A \cap B) = P(A) \times P(B)$$

Eşdeğer ifade: $P(A \mid B) = P(A)$ — B'yi bilmek A'yı etkilemez.

**Önemli:** Bağımsız ≠ Ayrık (karşılıklı dışlayıcı)

In [ ]:
def bagimsizilik_kontrol(S_kume, A_kume, B_kume, isim_A="A", isim_B="B"):
    """P(A∩B) ?= P(A)×P(B) kontrolü"""
    n = len(S_kume)
    p_A   = len(A_kume) / n
    p_B   = len(B_kume) / n
    p_AnB = len(A_kume & B_kume) / n
    p_carp = p_A * p_B
    bagims = abs(p_AnB - p_carp) < 1e-9
    print(f"  P({isim_A})={p_A:.4f}, P({isim_B})={p_B:.4f}")
    print(f"  P({isim_A}∩{isim_B})={p_AnB:.4f},  P({isim_A})×P({isim_B})={p_carp:.4f}")
    print(f"  Bağımsız mı? {'✓ EVET' if bagims else '✗ HAYIR'}")
    return bagims

S = {(i,j) for i in range(1,7) for j in range(1,7)}
print("=" * 52)
print("Bağımsızlık Örnekleri — İki Zar")
print("=" * 52)

# 1. İki bağımsız zar atışı — 1. atış çift, 2. atış çift
A1 = {(i,j) for i,j in S if i % 2 == 0}
B1 = {(i,j) for i,j in S if j % 2 == 0}
print("\n1. A={1. atış çift}, B={2. atış çift}")
bagimsizilik_kontrol(S, A1, B1, "1.çift", "2.çift")

# 2. Toplam 7 ve 1. atış 4 (bu da bağımsız)
A2 = {(i,j) for i,j in S if i+j == 7}
B2 = {(i,j) for i,j in S if i == 4}
print("\n2. A={Toplam 7}, B={1. atış 4}")
bagimsizilik_kontrol(S, A2, B2, "Toplam7", "1.atış4")

# 3. Ayrık ama bağımlı
A3 = {(i,j) for i,j in S if i == 1}
B3 = {(i,j) for i,j in S if i == 2}
print("\n3. A={1. atış=1}, B={1. atış=2} — AYRIK (karşılıklı dışlayıcı)")
bagimsizilik_kontrol(S, A3, B3, "1.atış=1", "1.atış=2")
print(f"  → Ayrık olaylar P(A)>0,P(B)>0 ise HİÇBİR ZAMAN bağımsız değildir!")

print()
# Yazı-tura: 5 ardışık tura
p_5_yazi = (1/2)**5
print(f"Bağımsızlık uygulaması: P(5 yazı art arda) = (1/2)⁵ = {p_5_yazi:.5f} = {Fraction(1,32)}")

In [ ]:
# Bağımsızlık özelliklerini doğrulama
print("Bağımsızlık Özellikleri")
print("=" * 48)

# Adil tek zar
S_tek = set(range(1, 7))
A = {2, 4, 6}    # çift
B = {1, 2, 3}    # ≤ 3

Ac = S_tek - A
Bc = S_tek - B

def p_tek(E): return len(E) / len(S_tek)
def bag(X, Y): return abs(p_tek(X&Y) - p_tek(X)*p_tek(Y)) < 1e-9

print(f"A = {sorted(A)} (çift),  P(A) = {p_tek(A):.4f}")
print(f"B = {sorted(B)} (≤3),    P(B) = {p_tek(B):.4f}")
print()
print(f"A ve B  bağımsız mı?  {bag(A,B)}  → P(A∩B)={p_tek(A&B):.4f}, P(A)P(B)={p_tek(A)*p_tek(B):.4f}")
print(f"A ve Bᶜ bağımsız mı?  {bag(A,Bc)} → P(A∩Bᶜ)={p_tek(A&Bc):.4f}, P(A)P(Bᶜ)={p_tek(A)*p_tek(Bc):.4f}")
print(f"Aᶜ ve B bağımsız mı?  {bag(Ac,B)} → P(Aᶜ∩B)={p_tek(Ac&B):.4f}, P(Aᶜ)P(B)={p_tek(Ac)*p_tek(B):.4f}")
print(f"Aᶜ ve Bᶜ bağımsız mı? {bag(Ac,Bc)} → P(Aᶜ∩Bᶜ)={p_tek(Ac&Bc):.4f}, P(Aᶜ)P(Bᶜ)={p_tek(Ac)*p_tek(Bc):.4f}")
print()
print("Sonuç: A ve B bağımsızsa → A ve Bᶜ, Aᶜ ve B, Aᶜ ve Bᶜ de bağımsız! ✓")

---
## 6. Karşılıklı Bağımsızlık

**İkili bağımsız:** Her iki olayın çifti bağımsız: $P(A_i \cap A_j) = P(A_i)P(A_j)$

**Karşılıklı bağımsız:** İkili bağımsızlık **ve** $P(A \cap B \cap C) = P(A)P(B)P(C)$

**İkili bağımsız ≠ Karşılıklı bağımsız!**

In [ ]:
# İki Zar: A={Toplam 7}, B={İlk 4}, C={İkinci 3}
# İkili bağımsız ama karşılıklı bağımsız DEĞİL

S2 = [(i,j) for i in range(1,7) for j in range(1,7)]
n2 = len(S2)

A = {(i,j) for i,j in S2 if i+j == 7}   # Toplam 7
B = {(i,j) for i,j in S2 if i == 4}      # İlk atış 4
C = {(i,j) for i,j in S2 if j == 3}      # İkinci atış 3

def p2(E): return len(E) / n2

print("İki Zar Örneği: A={Toplam 7}, B={İlk=4}, C={İkinci=3}")
print("=" * 55)
print(f"P(A)={p2(A):.4f}={Fraction(len(A),n2)}, P(B)={p2(B):.4f}={Fraction(len(B),n2)}, P(C)={p2(C):.4f}={Fraction(len(C),n2)}")
print()

# İkili bağımsızlık kontrolleri
ciftler = [(A, B, 'A', 'B'), (B, C, 'B', 'C'), (A, C, 'A', 'C')]
print("İkili Bağımsızlık:")
for X, Y, nx, ny in ciftler:
    p_XY   = p2(X & Y)
    p_X_pY = p2(X) * p2(Y)
    esit   = abs(p_XY - p_X_pY) < 1e-9
    print(f"  P({nx}∩{ny}) = {p_XY:.4f} = {Fraction(len(X&Y),n2)},  "
          f"P({nx})×P({ny}) = {p_X_pY:.4f}  → {'✓ Bağımsız' if esit else '✗ Bağımlı'}")

# Üçlü kontrol
p_ABC   = p2(A & B & C)
p_A_B_C = p2(A) * p2(B) * p2(C)
print()
print("Üçlü Bağımsızlık:")
print(f"  P(A∩B∩C) = {p_ABC:.4f} = {Fraction(len(A&B&C),n2)},  P(A)×P(B)×P(C) = {p_A_B_C:.4f} = {Fraction(1,216)}")
esit3 = abs(p_ABC - p_A_B_C) < 1e-9
print(f"  → {'✓ Karşılıklı bağımsız' if esit3 else '✗ Karşılıklı bağımsız DEĞİL'}")
print()
print("Neden? A∩B∩C = {(4,3)} — Toplam 7 VE 1.atış=4 VE 2.atış=3 aynı anda gerektirir")
print("Bu üç koşul birbiriyle mantıksal olarak bağlantılı → Karşılıklı bağımsızlık bozuluyor")

In [ ]:
# Bağımsız olayların uygulaması — sistem güvenilirliği
print("Uygulama: Sistem Güvenilirliği")
print("=" * 48)
print("Her bileşen bağımsız olarak P(çalışıyor) = 0.95")

p_bilesken = 0.95

# Seri bağlantı: P(sistem) = Π P(bileşen)
print("\n1. Seri Bağlantı (tüm bileşenler çalışmalı):")
for n in [1, 2, 3, 5, 10]:
    p_seri = p_bilesken ** n
    print(f"   n={n:2d} bileşen: P(sistem) = 0.95^{n} = {p_seri:.5f}")

# Paralel bağlantı: P(sistem çalışmaz) = Π P(bileşen çalışmaz)
print("\n2. Paralel Bağlantı (en az biri çalışmalı):")
for n in [1, 2, 3, 5, 10]:
    p_paralel = 1 - (1 - p_bilesken) ** n
    print(f"   n={n:2d} bileşen: P(sistem) = 1 - 0.05^{n} = {p_paralel:.7f}")

---
## 7. Olasılık Algıları ve Önyargılar — Linda Problemi

**Birleşim önyargısı (conjunction fallacy):** İnsanlar $P(A \cap B) > P(B)$ olduğunu düşünme eğilimindedir.
Oysa matematiksel gerçek: $P(A \cap B) \leq P(B)$ her zaman!

Tversky ve Kahneman (1983) tarafından keşfedilmiştir.

In [ ]:
# Linda problemi — matematiksel analiz
print("Linda Problemi — Matematiksel Analiz")
print("=" * 52)

# Varsayımsal olasılıklar
p_A  = 0.70   # P(Feminist harekette aktif)
p_B  = 0.15   # P(Banka veznedarı)
# P(A∩B) için değişik senaryolar

print(f"A = {{Linda feminist harekette aktiftir}},  P(A) = {p_A}")
print(f"B = {{Linda banka veznedarıdır}},           P(B) = {p_B}")
print()
print(f"MATEMATİKSEL KISIT: P(A∩B) ≤ min(P(A), P(B)) = {min(p_A, p_B)}")
print()

# A ve B bağımsızsa (üst sınır senaryosu)
p_AnB_bag = p_A * p_B
print(f"Bağımsızlık varsayımıyla:  P(A∩B) = {p_AnB_bag:.4f}")
print(f"Her zaman: P(A∩B)={p_AnB_bag:.4f} ≤ P(B)={p_B:.4f}  → {'✓' if p_AnB_bag <= p_B else '✗'}")
print()

# Kahneman-Tversky deneyinin sayısal sonuçları
print("Tarihsel Deney Sonuçları (Tversky & Kahneman, 1983):")
print("-" * 52)
sonuclar = [
    ('Linda feminist harekette aktiftir', 'A', 2.1),
    ('Linda banka veznedarı ve feministtir', 'A∩B', 4.1),
    ('Linda banka veznedarıdır', 'B', 6.2),
]
print(f"{'İfade':<45} {'Olay':>5} {'Ort. Sıra':>10}")
print("-" * 63)
for ifade, olay, sira in sonuclar:
    isaretler = " ← DAHA DÜŞÜK sıra = DAHA OLASI" if olay == 'A∩B' else ""
    print(f"{ifade:<45} {olay:>5} {sira:>10.1f}{isaretler}")
print()
print("PROBLEM: Katılımcıların %90'ı 'banka veznedarı' yerine")
print("         'banka veznedarı VE feminist'i daha olası seçti.")
print(f"         Oysa: P(A∩B) ≤ P(B) HER ZAMAN!")

In [ ]:
# Linda problemi görselleştirmesi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: Matematiksel gerçek — P(A∩B) ≤ P(A), P(A∩B) ≤ P(B)
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Matematiksel Gerçek: P(A∩B) ≤ P(B)', fontsize=11, fontweight='bold')

rect = plt.Rectangle((0.2, 0.5), 9.6, 5, fill=False, edgecolor='black', linewidth=2)
ax.add_patch(rect)
cA = Circle((3.8, 3.0), 2.2, color='#92c5de', alpha=0.55, zorder=2)
cB = Circle((6.5, 3.0), 2.0, color='#f4a582', alpha=0.55, zorder=2)
for c in [cA, cB]: ax.add_patch(c)
ax.add_patch(Circle((3.8,3.0),2.2,fill=False,ec='#2166ac',lw=2,zorder=3))
ax.add_patch(Circle((6.5,3.0),2.0,fill=False,ec='#ca0020',lw=2,zorder=3))
ax.text(2.5, 3.8, 'A\n(Feminist)', ha='center', fontsize=10, fontweight='bold', color='#2166ac')
ax.text(7.8, 3.8, 'B\n(Bankacı)', ha='center', fontsize=10, fontweight='bold', color='#ca0020')
ax.text(5.2, 3.0, 'A∩B', ha='center', fontsize=9, fontweight='bold', zorder=4)
ax.text(5, 0.2,
        'A∩B ⊆ B  →  P(A∩B) ≤ P(B)',
        ha='center', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange', lw=1.5))
ax.text(9.5, 5.3, 'S', fontsize=12, fontweight='bold')

# Sağ: Deney sonuçları karşılaştırması
ax2 = axes[1]
ifadeler = ['A\n(Feminist)', 'A∩B\n(Feminist\n+Bankacı)', 'B\n(Bankacı)']
siralama_kahneman = [2.1, 4.1, 6.2]
siralama_dogru    = [2.1, 6.5, 6.2]  # Matematiksel sıra (A∩B en az olası)

x = np.arange(len(ifadeler))
w = 0.35
b1 = ax2.bar(x - w/2, siralama_kahneman, w, label='Deney (insan)',
             color=['#4daf4a','#e41a1c','#984ea3'], alpha=0.85, edgecolor='black', lw=0.5)
b2 = ax2.bar(x + w/2, siralama_dogru, w, label='Matematiksel sıra',
             color=['#4daf4a','#aaaaaa','#984ea3'], alpha=0.5, edgecolor='black', lw=0.5,
             hatch='///')

for bar, v in zip(b1, siralama_kahneman):
    ax2.text(bar.get_x()+bar.get_width()/2, v+0.05, f'{v}', ha='center', va='bottom', fontsize=10)

ax2.set_xticks(x); ax2.set_xticklabels(ifadeler, fontsize=9)
ax2.set_ylabel('Ortalama Sıralama (1=en olası, 8=en az olası)', fontsize=9)
ax2.set_title('Linda Problemi: Deney vs Matematik\n(Düşük sıra = Daha olası)', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_ylim(0, 8)
ax2.grid(True, alpha=0.3, axis='y')
ax2.annotate('BİRLEŞİM\nÖNYARGISI!', xy=(1-w/2+0.05, 4.1), xytext=(1.8, 6.5),
             arrowprops=dict(arrowstyle='->', color='red', lw=2),
             fontsize=10, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 8. Özet

In [ ]:
print("=" * 68)
print("BÖLÜM 3 ÖZET — Koşullu Olasılık ve Bağımsızlık")
print("=" * 68)

ozet = [
    ("Koşullu Olasılık",     "P(A|B) = P(A∩B) / P(B)"),
    ("Sezgi",                "P(A|B) = |A∩B| / |B|  (eşit olasılıklı)"),
    ("Önerme 1",             "P(A|A) = 1"),
    ("Önerme 2",             "P(Aᶜ|B) = 1 − P(A|B)"),
    ("Çarpma Kuralı",        "P(A∩B) = P(A|B)·P(B)"),
    ("Zincir Kuralı",        "P(∩Aᵢ) = P(A₁)·P(A₂|A₁)·P(A₃|A₁,A₂)…"),
    ("Toplam Olas. Yasası",  "P(B) = Σ P(B|Aᵢ)·P(Aᵢ)  (bölümleme)"),
    ("Bayes Teoremi",        "P(Aᵢ|B) = P(B|Aᵢ)P(Aᵢ) / P(B)"),
    ("Önsel / Sonsal",       "P(Aᵢ): prior,  P(Aᵢ|B): posterior"),
    ("Bağımsızlık",          "P(A∩B) = P(A)·P(B)  ↔  P(A|B) = P(A)"),
    ("Bağımsız ≠ Ayrık",    "Ayrık olaylar genellikle bağımlıdır!"),
    ("Karşılıklı Bağımsız", "İkili bağımsızlık VE P(A∩B∩C)=P(A)P(B)P(C)"),
    ("Linda Problemi",       "P(A∩B) ≤ P(B) daima — birleşim önyargısı!"),
]

print(f"{'Kavram':<24} | {'Formül / Açıklama'}")
print("-" * 68)
for kavram, aciklama in ozet:
    print(f"{kavram:<24} | {aciklama}")
print("=" * 68)

---
## 9. Alıştırmalar

In [ ]:
# Soru 1: İki zar — toplam ≥ 10, ilk atış ≥ 4 koşulunda
print("SORU 1: P(Toplam ≥ 10 | İlk atış ≥ 4)")
S2 = {(i,j) for i in range(1,7) for j in range(1,7)}
A_s1 = {(i,j) for i,j in S2 if i+j >= 10}
B_s1 = {(i,j) for i,j in S2 if i >= 4}
cevap1 = kosullu_esit(S2, A_s1, B_s1)
print(f"  |A∩B| = {len(A_s1&B_s1)},  |B| = {len(B_s1)},  P(A|B) = {cevap1:.4f} = {Fraction(len(A_s1&B_s1),len(B_s1))}")

print()
# Soru 2: Toplam olasılık yasası
print("SORU 2: Fabrika üretimi — Makine A: %60 üretim, %3 bozuk")
print("        Makine B: %40 üretim, %5 bozuk. Rastgele ürün bozuk olma olasılığı?")
p_s2 = toplam_olasilik([0.60, 0.40], [0.03, 0.05])
print(f"  P(Bozuk) = 0.60×0.03 + 0.40×0.05 = {0.60*0.03:.4f} + {0.40*0.05:.4f} = {p_s2:.4f}")

print()
# Soru 3: Bayes
print("SORU 3: Bozuk ürün geldiğinde Makine A'dan gelme olasılığı?")
posterior_s3, _ = bayes([0.60, 0.40], [0.03, 0.05])
print(f"  P(Makine A | Bozuk) = {posterior_s3[0]:.4f} ≈ {posterior_s3[0]*100:.1f}%")

print()
# Soru 4: Bağımsızlık
print("SORU 4: P(A)=0.4, P(B)=0.3, P(A∩B)=0.12 ise A ve B bağımsız mı?")
pa, pb, panb = 0.4, 0.3, 0.12
print(f"  P(A)×P(B) = {pa*pb:.4f},  P(A∩B) = {panb:.4f}  → {'✓ Bağımsız' if abs(pa*pb-panb)<1e-9 else '✗ Bağımlı'}")